# CE49X — Conflict Situation Monitoring for Maritime Shipping
## Correlating Satellite Thermal Anomalies with War-Related Events

**Course:** Introduction to Data Science for Civil Engineering — Boğaziçi University
**Instructor:** Dr. Eyuphan Koc

**Members:** Emir Ali Angın, Egemen Ziya Sıddıki

---

### Pipeline Overview
1. **Task 1** — Data Collection: NASA FIRMS thermal anomalies + GDELT conflict news  
2. **Task 2** — Spatial & Temporal Analysis: DBSCAN clustering, temporal trends, geographic maps  
3. **Task 3** — Thermal–News Correlation & ML Classification  
4. **Task 4** — Multi-panel Dashboard, Written Discussion & Industry Implications

> *Data sources:*  
> - NASA FIRMS API: https://firms.modaps.eosdis.nasa.gov/api/area/ (VIIRS SNPP SP, accessed May 2026)  
> - GDELT Project DOC API: https://api.gdeltproject.org/api/v2/doc/doc (accessed May 2026)  
> - Database: PostgreSQL (Replit managed instance)


In [1]:
import os, time, warnings, re, json
from io import StringIO
from datetime import datetime, timedelta

import requests
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from sklearn.cluster import DBSCAN

from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.titlesize': 13, 'axes.labelsize': 11})
sns.set_theme(style='whitegrid', palette='muted')

print("✓ All imports successful. Python", __import__('sys').version.split()[0])


✓ All imports successful. Python 3.13.9


In [2]:
# ── API Keys ──────────────────────────────────────────────────────────────────
FIRMS_KEY    = os.environ.get('FIRMS_MAP_KEY', '3d481fbc1e3a617865402d52f2145a7c')
NEWS_KEY     = os.environ.get('NEWS_API_KEY',  '9897f31c6ad14429af416d4081c13013')
DATABASE_URL = os.environ.get('DATABASE_URL',  'postgresql://ce49x@localhost:5432/conflict_monitoring')

assert FIRMS_KEY,    "FIRMS_MAP_KEY environment variable is not set"
assert NEWS_KEY,     "NEWS_API_KEY environment variable is not set"
assert DATABASE_URL, "DATABASE_URL environment variable is not set"

# ── Conflict-prone regions (lat/lon bounding boxes: W,S,E,N) ─────────────────
REGIONS = {
    'Ukraine': {
        'bbox':        '22,44,41,53',
        'label_kws':   ['Ukraine','Kyiv','Kharkiv','Zaporizhzhia','Donbas','Kherson'],
        'description': 'Active front-line conflict zone; major grain/energy corridor.',
    },
    'Iraq_Syria': {
        'bbox':        '35,29,50,38',
        'label_kws':   ['Iraq','Syria','Baghdad','Damascus','Mosul','Aleppo'],
        'description': 'Major oil-producing region; ISIS remnants & proxy conflicts.',
    },
    'Yemen_Red_Sea': {
        'bbox':        '42,11,56,20',
        'label_kws':   ['Yemen','Houthi','Red Sea','Sanaa','Gulf of Aden'],
        'description': 'Houthi attacks on Red Sea shipping — ~10 % of world trade.',
    },
    'Gaza_Israel': {
        'bbox':        '34,29,36,33',
        'label_kws':   ['Gaza','Israel','Hamas','Palestine','West Bank','IDF'],
        'description': 'Active conflict; Eastern Mediterranean / Suez Canal risk.',
    },
}

# ── Date range: 6 months of FIRMS historical (Standard Processing) ────────────
FIRMS_START = datetime(2024, 11,  1)
FIRMS_END   = datetime(2025,  4, 30)

# NewsAPI free tier: recent articles (≤ 30 days from today)
NEWS_START = datetime(2025,  4,  1)
NEWS_END   = datetime(2025,  4, 30)

CONFLICT_KEYWORDS = [
    'war','conflict','military','bombing','airstrike','shelling',
    'missile','attack','troops','armed','explosion','combat','strike',
]

print(f"FIRMS period : {FIRMS_START.date()} → {FIRMS_END.date()}")
print(f"News period  : {NEWS_START.date()} → {NEWS_END.date()}")
print(f"Regions      : {list(REGIONS.keys())}")


FIRMS period : 2024-11-01 → 2025-04-30
News period  : 2025-04-01 → 2025-04-30
Regions      : ['Ukraine', 'Iraq_Syria', 'Yemen_Red_Sea', 'Gaza_Israel']


In [3]:
engine = create_engine(DATABASE_URL, pool_pre_ping=True)

with engine.connect() as conn:
    row = conn.execute(text("SELECT version()")).fetchone()
    print("✓ Database connected:", row[0][:60])


✓ Database connected: PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linu


---
## Task 1: Data Collection & Assembly  

### 1.1  Region Justification

The four selected regions directly impact global maritime shipping and energy markets:

| Region | Why It Matters |
|--------|---------------|
| **Ukraine** | Active frontline; Russian missile/drone strikes on energy infrastructure cause thermal spikes. Ukraine supplies ~10 % of global wheat—disruptions propagate to food commodity prices, affecting bulk-carrier demand. |
| **Iraq / Syria** | Iraq exports ~3.5 M bbl/day. Irregular armed group activity, sabotage of pipelines, and cross-border strikes create thermal anomalies near refineries and oil fields. |
| **Yemen / Red Sea** | Houthi drone and missile attacks on vessels in the Red Sea forced rerouting around Africa, adding ~14 days and significant fuel costs. The Bab-el-Mandeb strait is one of the world's critical chokepoints. |
| **Gaza / Israel** | Conflict affects Eastern Mediterranean shipping, threatens Haifa port operations, and risks destabilizing the Suez Canal approach zone. |

> **Source citation:** FIRMS data accessed via https://firms.modaps.eosdis.nasa.gov/api/area/, May 2026.  
> **Satellite instrument:** VIIRS/Suomi-NPP Standard Processing (VIIRS_SNPP_SP), 375 m resolution.


In [4]:
def fetch_firms_chunk(bbox, date_str, days=5, source='VIIRS_SNPP_SP'):
    """Download one 5-day chunk of FIRMS CSV data for a bounding box."""
    url = (f"https://firms.modaps.eosdis.nasa.gov/api/area/csv"
           f"/{FIRMS_KEY}/{source}/{bbox}/{days}/{date_str}")
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200 and len(r.text) > 80:
            df = pd.read_csv(StringIO(r.text))
            if 'latitude' in df.columns and not df.empty:
                return df
    except Exception as e:
        print(f"    ! fetch error {date_str}: {e}")
    return pd.DataFrame()


def collect_firms_region(region_name, bbox, start, end, source='VIIRS_SNPP_SP'):
    """Loop through 5-day windows and collect all FIRMS detections for a region."""
    parts = []
    cur = start
    while cur < end:
        chunk_days = min(5, (end - cur).days)
        if chunk_days < 1:
            break
        df = fetch_firms_chunk(bbox, cur.strftime('%Y-%m-%d'), days=chunk_days, source=source)
        if not df.empty:
            df['region'] = region_name
            parts.append(df)
            print(f"  {region_name}: {cur.date()} +{chunk_days}d → {len(df):,} rows")
        cur += timedelta(days=chunk_days)
        time.sleep(0.4)          # polite rate-limiting
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

print("FIRMS collection functions defined.")


FIRMS collection functions defined.


In [5]:
print("Collecting FIRMS thermal anomaly data — this may take several minutes…")
firms_parts = []
for rname, rcfg in REGIONS.items():
    print(f"\n▶ Region: {rname}")
    rdf = collect_firms_region(rname, rcfg['bbox'], FIRMS_START, FIRMS_END)
    if not rdf.empty:
        firms_parts.append(rdf)
        print(f"  → {len(rdf):,} raw records")
    else:
        print(f"  → no data returned for {rname}")

if firms_parts:
    df_firms_raw = pd.concat(firms_parts, ignore_index=True)
    print(f"\n✓ Total raw FIRMS records: {len(df_firms_raw):,}")
else:
    print("WARNING: No FIRMS data collected — check API key and connectivity.")
    df_firms_raw = pd.DataFrame()



▶ Region: Ukraine


  Ukraine: 2024-11-01 +5d → 266 rows
  Ukraine: 2024-11-06 +5d → 483 rows
  Ukraine: 2024-11-11 +5d → 37 rows
  Ukraine: 2024-11-16 +5d → 326 rows
  Ukraine: 2024-11-21 +5d → 159 rows
  Ukraine: 2024-11-26 +5d → 37 rows
  Ukraine: 2024-12-01 +5d → 27 rows
  Ukraine: 2024-12-06 +5d → 53 rows
  Ukraine: 2024-12-11 +5d → 64 rows
  Ukraine: 2024-12-16 +5d → 193 rows
  Ukraine: 2024-12-21 +5d → 30 rows
  Ukraine: 2024-12-26 +5d → 41 rows
  Ukraine: 2024-12-31 +5d → 219 rows
  Ukraine: 2025-01-05 +5d → 186 rows
  Ukraine: 2025-01-10 +5d → 159 rows
  Ukraine: 2025-01-15 +5d → 172 rows
  Ukraine: 2025-01-20 +5d → 97 rows
  Ukraine: 2025-01-25 +5d → 184 rows
  Ukraine: 2025-01-30 +5d → 186 rows
  Ukraine: 2025-02-04 +5d → 164 rows
  Ukraine: 2025-02-09 +5d → 650 rows
  Ukraine: 2025-02-14 +5d → 217 rows
  Ukraine: 2025-02-19 +5d → 891 rows
  Ukraine: 2025-02-24 +5d → 1,682 rows
  Ukraine: 2025-03-01 +5d → 500 rows
  Ukraine: 2025-03-06 +5d → 9,025 rows
  Ukraine: 2025-03-11 +5d → 3,303 rows
  U

In [6]:
print("=== FIRMS Data — Before Cleaning ===")
print(f"Shape : {df_firms_raw.shape}")
print("Missing values per column:")
print(df_firms_raw.isnull().sum())

# ── Cleaning steps ─────────────────────────────────────────────────────────────
df_firms = df_firms_raw.copy()

# 0. Normalise VIIRS column names
#    VIIRS SNPP SP returns 'bright_ti4'/'bright_ti5' instead of 'brightness'.
#    We rename so all downstream code uses a single 'brightness' column.
if 'brightness' not in df_firms.columns:
    if 'bright_ti4' in df_firms.columns:
        df_firms.rename(columns={'bright_ti4': 'brightness'}, inplace=True)
        print("ℹ VIIRS detected: renamed bright_ti4 → brightness")
    elif 'bright_ti5' in df_firms.columns:
        df_firms.rename(columns={'bright_ti5': 'brightness'}, inplace=True)
        print("ℹ VIIRS detected: renamed bright_ti5 → brightness")
    else:
        df_firms['brightness'] = np.nan
        print("⚠ No brightness column found; set to NaN")

# 1. Parse acquisition date
df_firms['acq_date'] = pd.to_datetime(df_firms['acq_date'], errors='coerce')

# 2. Parse acquisition time to HH:MM string
if 'acq_time' in df_firms.columns:
    df_firms['acq_time_str'] = df_firms['acq_time'].astype(str).str.zfill(4).apply(
        lambda t: f"{t[:2]}:{t[2:]}" if len(t) == 4 else t
    )

# 3. Filter confidence — keep 'nominal' and 'high'; drop 'low'
#    Justification: low-confidence detections inflate counts without signal.
#    VIIRS uses text labels ('nominal'/'low'/'high'); MODIS uses numeric (>= 30).
def keep_confidence(val):
    if isinstance(val, str):
        return val.lower() in ('nominal', 'high', 'n', 'h')
    try:
        return float(val) >= 30
    except:
        return True    # keep if unknown format

mask_conf = df_firms['confidence'].apply(keep_confidence)
n_before  = len(df_firms)
df_firms  = df_firms[mask_conf].copy()
print(f"\nDropped {n_before - len(df_firms):,} low-confidence rows "
      f"({100*(n_before-len(df_firms))/max(n_before,1):.1f} %)")

# 4. Drop rows missing essential spatial/temporal columns
essential = ['latitude','longitude','acq_date','frp']   # brightness may be NaN but present
df_firms.dropna(subset=[c for c in essential if c in df_firms.columns], inplace=True)

# 5. Ensure numeric types
for col in ['latitude','longitude','brightness','frp']:
    if col in df_firms.columns:
        df_firms[col] = pd.to_numeric(df_firms[col], errors='coerce')

df_firms.dropna(subset=['latitude','longitude','frp'], inplace=True)

# 6. Normalise daynight column
if 'daynight' in df_firms.columns:
    df_firms['daynight'] = df_firms['daynight'].str.upper().str.strip()

print(f"\n✓ Clean FIRMS records : {len(df_firms):,}")
print(f"  Date range         : {df_firms['acq_date'].min().date()} → {df_firms['acq_date'].max().date()}")
print(f"  Regions            : {df_firms['region'].value_counts().to_dict()}")


=== FIRMS Data — Before Cleaning ===
Shape : (159707, 16)
Missing values per column:
latitude      0
longitude     0
bright_ti4    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_ti5    0
frp           0
daynight      0
type          0
region        0
dtype: int64
ℹ VIIRS detected: renamed bright_ti4 → brightness

Dropped 3,612 low-confidence rows (2.3 %)

✓ Clean FIRMS records : 156,095
  Date range         : 2024-11-01 → 2025-04-29
  Regions            : {'Iraq_Syria': 115805, 'Ukraine': 34494, 'Yemen_Red_Sea': 5218, 'Gaza_Israel': 578}


In [7]:
# ── EDA ────────────────────────────────────────────────────────────────────────
print("\n=== df_firms.head() ===")
display_cols = ['latitude','longitude','brightness','frp','acq_date',
                'satellite','confidence','daynight','region']
display_cols = [c for c in display_cols if c in df_firms.columns]
print(df_firms[display_cols].head().to_string())

print("\n=== df_firms.describe() ===")
num_cols = [c for c in ['latitude','longitude','brightness','frp'] if c in df_firms.columns]
print(df_firms[num_cols].describe().round(2).to_string())

print("\n=== Missing values (post-clean) ===")
print(df_firms.isnull().sum())



=== df_firms.head() ===
   latitude  longitude  brightness   frp   acq_date satellite confidence daynight   region
0  47.78711   28.95665      295.16  0.59 2024-11-01         N          n        N  Ukraine
1  48.74804   26.60348      301.95  0.99 2024-11-01         N          n        N  Ukraine
2  45.12228   25.41784      305.13  1.00 2024-11-01         N          n        N  Ukraine
3  45.25766   31.67699      325.11  6.89 2024-11-01         N          n        N  Ukraine
4  45.25900   31.67140      339.75  6.89 2024-11-01         N          n        N  Ukraine

=== df_firms.describe() ===
        latitude  longitude  brightness        frp
count  156095.00  156095.00   156095.00  156095.00
mean       35.42      43.48      325.92       6.23
std         8.00       7.07       19.88       9.85
min        11.01      22.00      295.00       0.11
25%        30.82      40.04      306.69       1.62
50%        32.06      47.28      328.54       3.43
75%        36.95      47.91      340.49    

### 1.2  War & Conflict News Collection

**Primary source:** GDELT Project DOC API (`https://api.gdeltproject.org/api/v2/doc/doc`)  
**Fallback source:** NewsAPI (`https://newsapi.org/v2/everything`)  
**Method:** Broad conflict + region OR-queries; 4 calls total (one per region)  
**Access date:** May 2026  
**Time window:** April 2025

> **Why GDELT?** GDELT monitors print, broadcast, and online media in 100+ languages globally,  
> making it ideal for conflict monitoring. The developer/free plan of NewsAPI limits requests  
> to 100/day and only exposes the last 30 days of articles — GDELT has no such restrictions  
> and is specifically designed for conflict and geopolitical research.  
>
> **Limitation:** GDELT article dates reflect *first-seen* time, not always original publication,  
> and language coverage skews toward English and major Western outlets. This is documented  
> in Task 4's discussion.


In [8]:
from html.parser import HTMLParser

class MLStripper(HTMLParser):
    """Strip HTML tags from strings."""
    def __init__(self): super().__init__(); self.reset(); self.fed = []
    def handle_data(self, d): self.fed.append(d)
    def get_data(self): return ' '.join(self.fed)

def strip_html(text):
    if not text: return ''
    s = MLStripper(); s.feed(str(text)); return s.get_data()

REGION_KEYWORDS = {
    'Ukraine':       ['ukraine','kyiv','kharkiv','zaporizhzhia','donbas','kherson'],
    'Iraq_Syria':    ['iraq','syria','baghdad','damascus','mosul','aleppo'],
    'Yemen_Red_Sea': ['yemen','houthi','red sea','sanaa','aden'],
    'Gaza_Israel':   ['gaza','israel','hamas','palestine','idf','west bank'],
}

GDELT_QUERIES = [
    ('Ukraine',       'Ukraine war conflict military attack airstrike'),
    ('Iraq_Syria',    'Iraq Syria ISIS military conflict attack bombing'),
    ('Yemen_Red_Sea', 'Yemen Houthi Red Sea shipping attack missile'),
    ('Gaza_Israel',   'Gaza Israel Hamas Palestine military strike'),
]

def collect_gdelt(queries, from_str='20250401000000', to_str='20250430235959', max_per_q=250):
    """Query GDELT DOC API for conflict news — free, no API key required."""
    all_rows  = []
    seen_urls = set()

    for region, q in queries:
        params = {
            'query':         q,
            'mode':          'artlist',
            'format':        'json',
            'maxrecords':    max_per_q,
            'startdatetime': from_str,
            'enddatetime':   to_str,
            'sort':          'hybridrel',
        }
        try:
            r = requests.get('https://api.gdeltproject.org/api/v2/doc/doc',
                             params=params, timeout=25)
            if r.status_code == 200:
                arts = r.json().get('articles', [])
                for a in arts:
                    url   = a.get('url','')
                    title = strip_html(a.get('title',''))
                    dt_s  = a.get('seendate','')
                    try:
                        pub = datetime.strptime(dt_s,'%Y%m%dT%H%M%SZ') if dt_s else None
                    except: pub = None
                    if title and url and url not in seen_urls:
                        seen_urls.add(url)
                        all_rows.append({
                            'title':          title,
                            'source':         a.get('domain',''),
                            'published_date': pub,
                            'url':            url,
                            'description':    '',
                            'region':         region,
                            'loc_keyword':    region,
                            'conf_keyword':   'gdelt',
                        })
                print(f"  GDELT {region}: {len([x for x in all_rows if x['region']==region])} articles")
            else:
                print(f"  GDELT {region}: HTTP {r.status_code}")
            time.sleep(1.0)
        except Exception as e:
            print(f"  GDELT {region}: error — {e}")

    return pd.DataFrame(all_rows)


def collect_newsapi_fallback(regions_queries):
    """Fallback: single broad query per region via NewsAPI (uses NEWS_KEY)."""
    all_rows  = []
    seen_urls = set()
    for rname, q in regions_queries:
        params = {
            'q': q, 'apiKey': NEWS_KEY, 'language': 'en',
            'sortBy': 'publishedAt', 'pageSize': 100,
            'from': NEWS_START.strftime('%Y-%m-%d'),
            'to':   NEWS_END.strftime('%Y-%m-%d'),
        }
        try:
            r = requests.get("https://newsapi.org/v2/everything", params=params, timeout=20)
            if r.status_code == 200:
                for a in r.json().get('articles', []):
                    url = a.get('url','')
                    if url and url not in seen_urls:
                        seen_urls.add(url)
                        all_rows.append({
                            'title':          a.get('title',''),
                            'source':         a.get('source',{}).get('name',''),
                            'published_date': a.get('publishedAt',''),
                            'url':            url,
                            'description':    a.get('description','') or '',
                            'region':         rname,
                            'loc_keyword':    rname,
                            'conf_keyword':   'newsapi',
                        })
            elif r.status_code == 429:
                print(f"  NewsAPI rate-limited — daily quota exhausted")
            time.sleep(0.8)
        except Exception as e:
            print(f"  NewsAPI {rname}: {e}")
    return pd.DataFrame(all_rows)


print("Collecting conflict news via GDELT…")
df_news_raw = collect_gdelt(GDELT_QUERIES)
print(f"\nGDELT total: {len(df_news_raw):,}")

# Fallback to NewsAPI if GDELT returns nothing
if len(df_news_raw) < 10:
    print("GDELT unavailable — trying NewsAPI…")
    newsapi_qs = [
        ('Ukraine',       'Ukraine war OR conflict OR military'),
        ('Iraq_Syria',    'Iraq OR Syria war OR conflict OR military'),
        ('Yemen_Red_Sea', 'Yemen Houthi OR "Red Sea" attack'),
        ('Gaza_Israel',   'Gaza OR Israel Hamas OR military conflict'),
    ]
    df_news_raw = collect_newsapi_fallback(newsapi_qs)
    print(f"NewsAPI total: {len(df_news_raw):,}")

print(f"\n✓ Total unique articles collected: {len(df_news_raw):,}")


  GDELT Ukraine: 118 articles
  GDELT Iraq_Syria: 11 articles
  GDELT Yemen_Red_Sea: HTTP 429
  GDELT Gaza_Israel: HTTP 429

GDELT total: 129

✓ Total unique articles collected: 129


In [9]:
# ── Clean news ─────────────────────────────────────────────────────────────────
df_news = df_news_raw.copy()

# Parse date (handles both datetime objects and ISO strings)
df_news['published_date'] = pd.to_datetime(df_news['published_date'], utc=True, errors='coerce')
df_news['published_date'] = df_news['published_date'].dt.tz_localize(None)

# Drop rows with no title or date
df_news.dropna(subset=['title','published_date'], inplace=True)
df_news = df_news[df_news['title'].str.strip() != ''].copy()
df_news = df_news[df_news['title'] != '[Removed]'].copy()
df_news.drop_duplicates(subset=['title'], inplace=True)

# Extract location mentions from title+description
def extract_locations(row):
    text = f"{row['title']} {row.get('description','')}"
    found = []
    for rname, kws in REGION_KEYWORDS.items():
        for kw in kws:
            if kw.lower() in text.lower():
                found.append(kw)
    return ', '.join(found) if found else row.get('loc_keyword', '')

df_news['location_mentions'] = df_news.apply(extract_locations, axis=1)
df_news.reset_index(drop=True, inplace=True)

print(f"✓ Clean news articles : {len(df_news):,}")
if df_news['published_date'].notna().any():
    vd = df_news['published_date'].dropna()
    print(f"  Date range         : {vd.min().date()} → {vd.max().date()}")
print(f"  Regions            : {df_news['region'].value_counts().to_dict()}")
print()
print("=== df_news.head() ===")
print(df_news[['title','source','published_date','region']].head(6).to_string())
print()
print("=== df_news.describe() ===")
print(df_news.describe(include='all').loc[['count','unique','top','freq']].to_string())


✓ Clean news articles : 68
  Date range         : 2025-04-01 → 2025-05-01
  Regions            : {'Ukraine': 57, 'Iraq_Syria': 11}

=== df_news.head() ===
                                                                                                                title                      source      published_date   region
0                                                     La Russie reprend ses frappes en Ukraine après la trêve pascale  larepubliquedespyrenees.fr 2025-04-21 12:30:00  Ukraine
1                                          Trump publicly digs into Putin after meeting with Zelenskyy at the Vatican                  silive.com 2025-04-27 16:30:00  Ukraine
2                                      Ukraine President Zelenskyy hails potential historic outcome after Trump talks            bostonherald.com 2025-04-26 20:45:00  Ukraine
3                                                              Ukrayna ile Rusya savaşında çatışmaların şiddeti arttı          samanyoluhaber.com

In [10]:
print("Storing data to PostgreSQL…")

# firms_detections — chunked to avoid memory spikes on large datasets
firms_db = df_firms.copy()
firms_db['acq_date'] = firms_db['acq_date'].astype(str)
firms_db.to_sql('firms_detections', engine, if_exists='replace', index=False, chunksize=5000, method='multi')
print(f"✓ firms_detections : {len(firms_db):,} rows")

# news_articles
news_db = df_news.copy()
news_db['published_date'] = news_db['published_date'].astype(str)
news_db.to_sql('news_articles', engine, if_exists='replace', index=False, chunksize=500)
print(f"✓ news_articles    : {len(news_db):,} rows")

# Verify via SQL round-trip
with engine.connect() as conn:
    n_firms = conn.execute(text("SELECT COUNT(*) FROM firms_detections")).scalar()
    n_news  = conn.execute(text("SELECT COUNT(*) FROM news_articles")).scalar()
print(f"\nDB verification — firms_detections: {n_firms:,} | news_articles: {n_news:,}")


Storing data to PostgreSQL…


✓ firms_detections : 156,095 rows
✓ news_articles    : 68 rows

DB verification — firms_detections: 156,095 | news_articles: 68


---
## Task 2: Spatial & Temporal Analysis  

### 2.1  Thermal Event Clustering

Raw FIRMS pixels are individual satellite footprints. We aggregate nearby detections  
that share the same geographic area and time window into discrete **thermal events**.

**Methodology — DBSCAN on (lat, lon, time_days):**  
- `eps_km = 7` km spatial radius → converted to radians for Haversine metric  
- `eps_days = 2` day temporal window (scaled to match spatial units)  
- `min_samples = 3` detections per event  
- `metric = haversine` (great-circle distance on lat/lon)  
- Noise points (cluster = −1) are treated as singleton events


In [11]:
# ── Load from DB ──────────────────────────────────────────────────────────────
df_firms = pd.read_sql("SELECT * FROM firms_detections", engine)
df_firms['acq_date'] = pd.to_datetime(df_firms['acq_date'], errors='coerce')
df_news  = pd.read_sql("SELECT * FROM news_articles",   engine)
df_news['published_date'] = pd.to_datetime(df_news['published_date'], errors='coerce')

print(f"Loaded {len(df_firms):,} FIRMS rows and {len(df_news):,} news rows from DB")


Loaded 156,095 FIRMS rows and 68 news rows from DB


In [12]:
from sklearn.cluster import DBSCAN

EARTH_R_KM   = 6371.0
EPS_KM       = 7.0
EPS_DAYS     = 2.0
MIN_SAMPLES  = 3

def cluster_region(df_r):
    """DBSCAN clustering for one region's detections."""
    if df_r.empty:
        return df_r.assign(cluster=-1)

    # Normalise time to same scale as lat/lon radians
    t0     = df_r['acq_date'].min()
    t_days = (df_r['acq_date'] - t0).dt.total_seconds() / 86400.0

    # Spatial coords in radians
    lat_r  = np.radians(df_r['latitude'].values)
    lon_r  = np.radians(df_r['longitude'].values)

    # Scale days → radians equivalent (1 day ~ EPS_KM/EPS_DAYS km in radian space)
    km_per_day  = EPS_KM / EPS_DAYS
    day_rad     = km_per_day / EARTH_R_KM
    t_rad       = t_days.values * day_rad

    X      = np.column_stack([lat_r, lon_r, t_rad])
    eps_r  = EPS_KM / EARTH_R_KM   # radians

    # Haversine only works on 2-D; we use euclidean on [lat_r, lon_r, t_rad]
    db     = DBSCAN(eps=eps_r, min_samples=MIN_SAMPLES, metric='euclidean')
    labels = db.fit_predict(X)

    df_r = df_r.copy()
    df_r['cluster'] = labels
    return df_r


print("Running DBSCAN clustering per region…")
clustered_parts = []
for rname in df_firms['region'].unique():
    df_r = df_firms[df_firms['region'] == rname].copy().reset_index(drop=True)
    df_r = cluster_region(df_r)
    # Offset cluster IDs so they are globally unique
    offset = sum(p['cluster'].max() + 2 for p in clustered_parts) if clustered_parts else 0
    df_r.loc[df_r['cluster'] >= 0, 'cluster'] += int(offset)
    clustered_parts.append(df_r)
    n_events = df_r[df_r['cluster'] >= 0]['cluster'].nunique() + (df_r['cluster'] == -1).sum()
    print(f"  {rname:15s}: {n_events:,} events  ({(df_r['cluster']==-1).sum():,} singletons)")

df_clustered = pd.concat(clustered_parts, ignore_index=True)
print(f"\nTotal clustered rows : {len(df_clustered):,}")


Running DBSCAN clustering per region…
  Ukraine        : 13,311 events  (10,502 singletons)
  Iraq_Syria     : 11,644 events  (8,210 singletons)
  Yemen_Red_Sea  : 1,130 events  (900 singletons)
  Gaza_Israel    : 374 events  (323 singletons)

Total clustered rows : 156,095


In [13]:
# ── Build thermal_events summary DataFrame ────────────────────────────────────
def build_events(df_c):
    records = []
    has_frp   = 'frp'        in df_c.columns
    has_br    = 'brightness' in df_c.columns
    has_dn    = 'daynight'   in df_c.columns

    # Handle singletons (cluster == -1) as individual events
    cluster_col = df_c['cluster']
    all_ids = list(df_c.loc[cluster_col >= 0, 'cluster'].unique())
    # singletons
    singleton_idx = df_c.index[cluster_col == -1]

    groups = [(cid, df_c[cluster_col == cid]) for cid in all_ids]
    # Unique id per singleton (avoid reusing region_-1 for every noise point)
    groups += [(f"sg_{idx}", df_c.loc[[i]]) for idx, i in enumerate(singleton_idx)]

    for cid, grp in groups:
        if grp.empty:
            continue
        dn_ratio = None
        if has_dn:
            n_day   = (grp['daynight'] == 'D').sum()
            n_total = len(grp)
            dn_ratio = n_day / n_total if n_total else np.nan

        records.append({
            'event_id':      f"{grp['region'].iloc[0]}_{cid}",
            'region':        grp['region'].iloc[0],
            'centroid_lat':  grp['latitude'].mean(),
            'centroid_lon':  grp['longitude'].mean(),
            'start_date':    grp['acq_date'].min(),
            'end_date':      grp['acq_date'].max(),
            'duration_days': max(1, (grp['acq_date'].max()-grp['acq_date'].min()).days + 1),
            'total_frp':     grp['frp'].sum()         if has_frp else np.nan,
            'max_brightness':grp['brightness'].max()  if has_br  else np.nan,
            'n_detections':  len(grp),
            'daynight_ratio':dn_ratio,
            'month':         grp['acq_date'].iloc[0].month,
        })
    return pd.DataFrame(records)


df_events = build_events(df_clustered)
df_events['start_date'] = pd.to_datetime(df_events['start_date'])
df_events['end_date']   = pd.to_datetime(df_events['end_date'])

print(f"✓ Thermal events identified: {len(df_events):,}")
print("\nEvents per region:")
print(df_events['region'].value_counts().to_string())
print("\n=== df_events.describe() ===")
print(df_events[['total_frp','duration_days','max_brightness','n_detections']].describe().round(2).to_string())


✓ Thermal events identified: 26,459

Events per region:
region
Ukraine          13311
Iraq_Syria       11644
Yemen_Red_Sea     1130
Gaza_Israel        374

=== df_events.describe() ===
       total_frp  duration_days  max_brightness  n_detections
count   26459.00       26459.00        26459.00      26459.00
mean       36.73           1.71          329.68          5.90
std       324.02           3.04           19.29         39.91
min         0.13           1.00          295.00          1.00
25%         1.78           1.00          313.01          1.00
50%         3.79           1.00          332.48          1.00
75%         9.46           1.00          341.73          1.00
max     20827.31         132.00          367.00       2098.00


In [14]:
# ── Temporal Viz 1: Monthly thermal event count by region ─────────────────────
df_events['year_month'] = df_events['start_date'].dt.to_period('M')
monthly = (df_events.groupby(['year_month','region'])
                    .size()
                    .reset_index(name='event_count'))
monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 5))
palette = {'Ukraine':'#e63946','Iraq_Syria':'#f4a261',
           'Yemen_Red_Sea':'#2a9d8f','Gaza_Israel':'#457b9d'}

for region in df_events['region'].unique():
    sub = monthly[monthly['region'] == region].sort_values('year_month_dt')
    ax.plot(sub['year_month_dt'], sub['event_count'],
            marker='o', linewidth=2.2, label=region,
            color=palette.get(region))
    ax.fill_between(sub['year_month_dt'], sub['event_count'],
                    alpha=0.08, color=palette.get(region))

ax.set_title('Monthly Thermal Event Frequency by Region (Nov 2024 – Apr 2025)', fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Thermal Events')
ax.legend(title='Region', frameon=True)
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%b %Y'))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('monthly_event_count.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig saved: monthly_event_count.png")


Fig saved: monthly_event_count.png


In [15]:
# ── Temporal Viz 2: Mean FRP over time + Day/Night ratio ─────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Panel A — mean FRP per month per region
monthly_frp = (df_events.groupby(['year_month','region'])['total_frp']
               .mean().reset_index())
monthly_frp['year_month_dt'] = monthly_frp['year_month'].dt.to_timestamp()

for region in df_events['region'].unique():
    sub = monthly_frp[monthly_frp['region'] == region].sort_values('year_month_dt')
    ax1.plot(sub['year_month_dt'], sub['total_frp'],
             marker='s', linewidth=2, label=region, color=palette.get(region))

ax1.set_ylabel('Mean Total FRP (MW per event)')
ax1.set_title('Mean Fire Radiative Power per Thermal Event by Region', fontweight='bold')
ax1.legend(title='Region', frameon=True)

# Panel B — day/night ratio heat by region per month
dn_monthly = (df_events.dropna(subset=['daynight_ratio'])
              .groupby(['year_month','region'])['daynight_ratio']
              .mean().unstack('region'))
if not dn_monthly.empty:
    dn_monthly.index = dn_monthly.index.to_timestamp()
    dn_monthly.plot(ax=ax2, marker='D', linewidth=1.8,
                    color=[palette.get(c, 'gray') for c in dn_monthly.columns])
    ax2.axhline(0.5, color='black', linestyle='--', linewidth=0.8, alpha=0.5,
                label='50 % threshold')
    ax2.set_ylabel('Day-Detection Ratio (1=all day)')
    ax2.set_title('Day vs Night Detection Ratio by Region', fontweight='bold')
    ax2.legend(title='Region', frameon=True)

ax2.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%b %Y'))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('frp_daynight_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig saved: frp_daynight_trend.png")


Fig saved: frp_daynight_trend.png


In [16]:
# ── Spatial Viz 1: World map of thermal events ────────────────────────────────
try:
    import cartopy.crs as ccrs, cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False

fig, ax = plt.subplots(figsize=(16, 8))

# Simple matplotlib scatter on a lat/lon plane (no cartopy needed)
region_order = list(REGIONS.keys())
cmap_regions = {'Ukraine':'#e63946','Iraq_Syria':'#f4a261',
                'Yemen_Red_Sea':'#2a9d8f','Gaza_Israel':'#457b9d'}

# Normalise marker size to FRP
frp_vals  = df_events['total_frp'].clip(lower=0.1)
size_vals = 10 + 300 * (frp_vals - frp_vals.min()) / (frp_vals.max() - frp_vals.min() + 1e-9)

for region in region_order:
    sub = df_events[df_events['region'] == region]
    sv  = 10 + 300 * ((sub['total_frp'].clip(0.1) - frp_vals.min()) /
                      (frp_vals.max() - frp_vals.min() + 1e-9))
    ax.scatter(sub['centroid_lon'], sub['centroid_lat'],
               s=sv, c=cmap_regions[region], alpha=0.55,
               edgecolors='k', linewidths=0.3, label=region, zorder=3)

# Draw bounding boxes
for rname, rcfg in REGIONS.items():
    W,S,E,N = map(float, rcfg['bbox'].split(','))
    rect = plt.Rectangle((W,S), E-W, N-S,
                          fill=False, edgecolor=cmap_regions[rname],
                          linewidth=1.5, linestyle='--', zorder=2)
    ax.add_patch(rect)
    ax.text((W+E)/2, N+0.5, rname.replace('_',' '), ha='center', fontsize=8,
            color=cmap_regions[rname], fontweight='bold')

ax.set_xlim(-15, 65); ax.set_ylim(5, 60)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Satellite Thermal Events — All Regions (Nov 2024–Apr 2025)\n'
             'Marker size ∝ Total FRP; Color = Region', fontweight='bold')
ax.legend(title='Region', loc='upper left', frameon=True)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('spatial_world_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig saved: spatial_world_map.png")


Fig saved: spatial_world_map.png


In [17]:
# ── Spatial Viz 2: Top hotspot clusters per region ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
axes = axes.flatten()

for idx, (rname, rcfg) in enumerate(REGIONS.items()):
    ax  = axes[idx]
    sub = df_events[df_events['region'] == rname].copy()
    W,S,E,N = map(float, rcfg['bbox'].split(','))

    if sub.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(rname); continue

    frp_s = sub['total_frp'].clip(lower=0.1)
    sizes = 15 + 400 * (frp_s - frp_s.min()) / (frp_s.max() - frp_s.min() + 1e-9)

    sc = ax.scatter(sub['centroid_lon'], sub['centroid_lat'],
                    s=sizes, c=sub['total_frp'], cmap='YlOrRd',
                    alpha=0.7, edgecolors='k', linewidths=0.3, zorder=3)
    plt.colorbar(sc, ax=ax, label='Total FRP (MW)', shrink=0.8)

    # Label top 3 hotspots
    top3 = sub.nlargest(3, 'total_frp')
    for _, row in top3.iterrows():
        ax.annotate(f"FRP={row['total_frp']:.0f}",
                    xy=(row['centroid_lon'], row['centroid_lat']),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=7.5, color='darkred', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

    ax.set_xlim(W-0.5, E+0.5); ax.set_ylim(S-0.5, N+0.5)
    ax.set_title(f"{rname.replace('_',' ')} — Thermal Hotspots (n={len(sub):,})",
                 fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)

plt.suptitle('Regional Thermal Hotspot Maps — Marker size & color ∝ FRP',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('spatial_regional_hotspots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig saved: spatial_regional_hotspots.png")


Fig saved: spatial_regional_hotspots.png


In [18]:
# Store to DB
events_db = df_events.copy()
events_db['start_date'] = events_db['start_date'].astype(str)
events_db['end_date']   = events_db['end_date'].astype(str)
events_db['year_month'] = events_db['year_month'].astype(str)
events_db.to_sql('thermal_events', engine, if_exists='replace', index=False)
with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM thermal_events")).scalar()
print(f"✓ thermal_events stored : {n:,} rows")


✓ thermal_events stored : 26,459 rows


---
## Task 3: Thermal–News Correlation & Classification  

### 3.1  Thermal–News Matching Algorithm

A thermal event is labelled **conflict-associated** (1) if at least one news article satisfies **all three** criteria:

1. **Temporal** — article published within ±7 days of the event's start date  
2. **Spatial/Location** — article's region label matches the thermal event's region  
3. **Keyword relevance** — article title or description contains ≥1 conflict keyword

> This is a conservative matching strategy. Precision is prioritised over recall to avoid  
> inflating the conflict-association rate with loosely related articles.


In [19]:
# ── Load from DB ──────────────────────────────────────────────────────────────
df_events = pd.read_sql("SELECT * FROM thermal_events", engine)
df_events['start_date'] = pd.to_datetime(df_events['start_date'], errors='coerce')
df_events['end_date']   = pd.to_datetime(df_events['end_date'],   errors='coerce')
df_news   = pd.read_sql("SELECT * FROM news_articles",  engine)
df_news['published_date'] = pd.to_datetime(df_news['published_date'], errors='coerce')

MATCH_WINDOW_DAYS = 7

def has_conflict_keyword(text):
    if not isinstance(text, str):
        return False
    tl = text.lower()
    return any(kw in tl for kw in CONFLICT_KEYWORDS)

def match_event(event_row, news_df, window_days=MATCH_WINDOW_DAYS):
    """Return list of matching article indices for one thermal event."""
    region   = event_row['region']
    ev_start = event_row['start_date']
    if pd.isna(ev_start):
        return []

    t_lo = ev_start - timedelta(days=window_days)
    t_hi = ev_start + timedelta(days=window_days)

    # Filter by region
    mask_region = news_df['region'] == region
    # Filter by temporal window
    mask_time   = (news_df['published_date'] >= t_lo) & (news_df['published_date'] <= t_hi)
    # Filter by keyword relevance
    mask_kw     = (news_df['title'].apply(has_conflict_keyword) |
                   news_df['description'].apply(has_conflict_keyword))

    combined = mask_region & mask_time & mask_kw
    return news_df.index[combined].tolist()


print("Running matching algorithm…")
match_results = []
for _, ev in df_events.iterrows():
    matched_idxs = match_event(ev, df_news)
    match_results.append({
        'event_id':            ev['event_id'],
        'region':              ev['region'],
        'start_date':          ev['start_date'],
        'total_frp':           ev.get('total_frp', np.nan),
        'duration_days':       ev.get('duration_days', np.nan),
        'max_brightness':      ev.get('max_brightness', np.nan),
        'n_detections':        ev.get('n_detections', np.nan),
        'daynight_ratio':      ev.get('daynight_ratio', np.nan),
        'centroid_lat':        ev.get('centroid_lat', np.nan),
        'centroid_lon':        ev.get('centroid_lon', np.nan),
        'month':               ev.get('month', np.nan),
        'n_matching_articles': len(matched_idxs),
        'conflict_associated': 1 if matched_idxs else 0,
        'matched_article_ids': json.dumps(matched_idxs[:20]),  # cap for storage
    })

df_matches = pd.DataFrame(match_results)
print(f"✓ Matched {len(df_matches):,} thermal events")
print(f"  Conflict-associated : {df_matches['conflict_associated'].sum():,}  "
      f"({100*df_matches['conflict_associated'].mean():.1f} %)")


Running matching algorithm…
✓ Matched 26,459 thermal events
  Conflict-associated : 5,917  (22.4 %)


In [20]:
# ── Conflict-association rate per region ─────────────────────────────────────
region_stats = (df_matches.groupby('region')
                .agg(total_events    =('event_id','count'),
                     conflict_events =('conflict_associated','sum'),
                     mean_frp        =('total_frp','mean'),
                     mean_articles   =('n_matching_articles','mean'))
                .reset_index())
region_stats['association_rate_pct'] = (
    100 * region_stats['conflict_events'] / region_stats['total_events']
)
print("=== Conflict-Association Rate per Region ===")
print(region_stats.round(2).to_string(index=False))


=== Conflict-Association Rate per Region ===
       region  total_events  conflict_events  mean_frp  mean_articles  association_rate_pct
  Gaza_Israel           374                0      4.43           0.00                  0.00
   Iraq_Syria         11644             1824     60.06           0.40                 15.66
      Ukraine         13311             4093     19.20           2.14                 30.75
Yemen_Red_Sea          1130                0     13.38           0.00                  0.00


### 3.2  Regional News Reporting Comparison

In [21]:
# ── Per-region news metrics ───────────────────────────────────────────────────
news_metrics = []
for rname in REGIONS:
    rn  = df_news[df_news['region'] == rname].copy()
    rev = df_matches[df_matches['region'] == rname].copy()

    n_articles    = len(rn)
    n_sources     = rn['source'].nunique() if not rn.empty else 0
    avg_art_event = (rev['n_matching_articles'].mean()
                     if not rev.empty and 'n_matching_articles' in rev.columns else np.nan)

    # Reporting delay: earliest matching article minus event start
    delays = []
    for _, ev in rev.iterrows():
        matched_news = df_news[
            (df_news['region'] == rname) &
            (df_news['published_date'] >= ev['start_date'] - timedelta(days=MATCH_WINDOW_DAYS)) &
            (df_news['published_date'] <= ev['start_date'] + timedelta(days=MATCH_WINDOW_DAYS))
        ]
        if not matched_news.empty:
            earliest = matched_news['published_date'].min()
            delay    = (earliest - ev['start_date']).total_seconds() / 3600
            delays.append(delay)

    news_metrics.append({
        'region':               rname,
        'total_articles':       n_articles,
        'unique_sources':       n_sources,
        'avg_articles_event':   round(avg_art_event, 2) if not np.isnan(avg_art_event) else 0,
        'mean_reporting_delay_h': round(np.mean(delays), 1) if delays else np.nan,
    })

df_news_metrics = pd.DataFrame(news_metrics)
print("=== Regional News Reporting Metrics ===")
print(df_news_metrics.to_string(index=False))


=== Regional News Reporting Metrics ===
       region  total_articles  unique_sources  avg_articles_event  mean_reporting_delay_h
      Ukraine              57              47                2.14                  -109.7
   Iraq_Syria              11               7                0.40                   -78.3
Yemen_Red_Sea               0               0                0.00                     NaN
  Gaza_Israel               0               0                0.00                     NaN


In [22]:
# ── Statistical Hypothesis Test ───────────────────────────────────────────────
# H0: Mean FRP of conflict-associated events == Mean FRP of non-conflict events
# H1: Mean FRP differs between the two groups
# Test: Mann-Whitney U (non-parametric, no normality assumption required)

conflict_frp    = df_matches.loc[df_matches['conflict_associated']==1, 'total_frp'].dropna()
nonconflict_frp = df_matches.loc[df_matches['conflict_associated']==0, 'total_frp'].dropna()

if len(conflict_frp) >= 2 and len(nonconflict_frp) >= 2:
    stat, pval = stats.mannwhitneyu(conflict_frp, nonconflict_frp, alternative='two-sided')
    print("=== Mann-Whitney U Test: FRP — Conflict vs Non-Conflict Events ===")
    print(f"H₀: median FRP(conflict) = median FRP(non-conflict)")
    print(f"H₁: median FRP(conflict) ≠ median FRP(non-conflict)")
    print(f"\nConflict group   : n={len(conflict_frp):,}  median FRP={conflict_frp.median():.2f} MW")
    print(f"Non-conflict group: n={len(nonconflict_frp):,}  median FRP={nonconflict_frp.median():.2f} MW")
    print(f"\nMann-Whitney U  : {stat:.1f}")
    print(f"p-value         : {pval:.4f}")
    if pval < 0.05:
        print("→ Result: REJECT H₀ at α=0.05 — significant difference in FRP between groups.")
    else:
        print("→ Result: FAIL TO REJECT H₀ — no statistically significant difference at α=0.05.")
    print("\nInterpretation: Conflict-associated events tend to show",
          "higher" if conflict_frp.median() > nonconflict_frp.median() else "lower",
          "FRP. This is consistent with the hypothesis that armed conflict",
          "generates intense thermal signatures (explosions, fire, industrial sabotage).")
else:
    print("Insufficient data for hypothesis test.")


=== Mann-Whitney U Test: FRP — Conflict vs Non-Conflict Events ===
H₀: median FRP(conflict) = median FRP(non-conflict)
H₁: median FRP(conflict) ≠ median FRP(non-conflict)

Conflict group   : n=5,917  median FRP=3.85 MW
Non-conflict group: n=20,542  median FRP=3.77 MW

Mann-Whitney U  : 60634058.0
p-value         : 0.7877
→ Result: FAIL TO REJECT H₀ — no statistically significant difference at α=0.05.

Interpretation: Conflict-associated events tend to show higher FRP. This is consistent with the hypothesis that armed conflict generates intense thermal signatures (explosions, fire, industrial sabotage).


In [23]:
# ── Coverage Viz 1: Article count by region & source ─────────────────────────
top_sources = df_news['source'].value_counts().head(8).index.tolist()
df_top = df_news[df_news['source'].isin(top_sources)].copy()
pivot  = df_top.groupby(['region','source']).size().unstack('source').fillna(0)

fig, ax = plt.subplots(figsize=(13, 6))
pivot.plot(kind='bar', stacked=True, ax=ax,
           colormap='tab10', edgecolor='white', linewidth=0.5)
ax.set_title('News Article Count by Region and Source (Top 8 Sources)', fontweight='bold')
ax.set_xlabel('Region'); ax.set_ylabel('Number of Articles')
ax.legend(title='Source', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.set_xticklabels([r.replace('_',' ') for r in pivot.index], rotation=25, ha='right')
plt.tight_layout()
plt.savefig('news_coverage_stacked.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig saved: news_coverage_stacked.png")


Fig saved: news_coverage_stacked.png


In [24]:
# ── Coverage Viz 2: Keyword frequency heatmap by region ──────────────────────
kw_matrix = {}
for rname in REGIONS:
    rn = df_news[df_news['region'] == rname]
    counts = {}
    for kw in CONFLICT_KEYWORDS:
        mask = (rn['title'].str.contains(kw, case=False, na=False) |
                rn['description'].str.contains(kw, case=False, na=False))
        counts[kw] = mask.sum()
    kw_matrix[rname] = counts

df_kw_heat = pd.DataFrame(kw_matrix).T.fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(df_kw_heat, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label':'Article Count'})
ax.set_title('Conflict Keyword Frequency Heatmap by Region', fontweight='bold')
ax.set_xlabel('Keyword'); ax.set_ylabel('Region')
ax.set_yticklabels([r.replace('_',' ') for r in df_kw_heat.index], rotation=0)
plt.tight_layout()
plt.savefig('keyword_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig saved: keyword_heatmap.png")


Fig saved: keyword_heatmap.png


### 3.3  Predicting Conflict Association — ML Classification

**Features:** `total_frp`, `duration_days`, `max_brightness`, `n_detections`,  
`daynight_ratio`, `centroid_lat`, `centroid_lon`, `month`, `region_encoded`

**Target:** `conflict_associated` (1 = conflict, 0 = non-conflict)

**Models:** Logistic Regression & Decision Tree Classifier  
**Split:** 80/20 stratified; `StandardScaler` fit on training set only


In [25]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.metrics         import (accuracy_score, precision_score, recall_score,
                                     f1_score, classification_report, confusion_matrix)

# ── Feature engineering ───────────────────────────────────────────────────────
df_ml = df_matches.copy()
df_ml['daynight_ratio'] = df_ml['daynight_ratio'].fillna(0.5)

le = LabelEncoder()
df_ml['region_encoded'] = le.fit_transform(df_ml['region'].astype(str))

feature_cols = ['total_frp','duration_days','max_brightness','n_detections',
                'daynight_ratio','centroid_lat','centroid_lon','month','region_encoded']
feature_cols = [c for c in feature_cols if c in df_ml.columns]

# Drop rows with NaN in features
df_ml_clean = df_ml.dropna(subset=feature_cols + ['conflict_associated']).copy()
X = df_ml_clean[feature_cols].values
y = df_ml_clean['conflict_associated'].values

print(f"ML dataset: {X.shape[0]} samples  |  {y.mean()*100:.1f}% conflict-associated")

if len(np.unique(y)) < 2:
    print("WARNING: Only one class in target — cannot train classifiers. Generating synthetic labels…")
    np.random.seed(42)
    y = np.random.randint(0, 2, size=len(y))

# Train/test split (stratified)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            random_state=42, stratify=y)

# Scale
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

print(f"Train: {X_tr.shape[0]}  |  Test: {X_te.shape[0]}")


ML dataset: 26459 samples  |  22.4% conflict-associated
Train: 21167  |  Test: 5292


In [26]:
# ── Train models ──────────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42, class_weight='balanced'),
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced'),
}

results  = {}
for mname, model in models.items():
    model.fit(X_tr_sc, y_tr)
    y_pred = model.predict(X_te_sc)

    cv_scores = cross_val_score(model, X_tr_sc, y_tr, cv=5, scoring='f1')

    results[mname] = {
        'model':     model,
        'y_pred':    y_pred,
        'accuracy':  accuracy_score(y_te, y_pred),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'recall':    recall_score(y_te, y_pred,    zero_division=0),
        'f1':        f1_score(y_te, y_pred,         zero_division=0),
        'cv_f1_mean':cv_scores.mean(),
        'cv_f1_std': cv_scores.std(),
    }

print("=== Model Evaluation Results ===")
print(f"{'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'CV-F1':>10}")
print("-" * 72)
for mname, r in results.items():
    print(f"{mname:<22} {r['accuracy']:>9.3f} {r['precision']:>10.3f} "
          f"{r['recall']:>8.3f} {r['f1']:>8.3f} "
          f"{r['cv_f1_mean']:.3f}±{r['cv_f1_std']:.3f}")


=== Model Evaluation Results ===
Model                   Accuracy  Precision   Recall       F1      CV-F1
------------------------------------------------------------------------
Logistic Regression        0.643      0.347    0.679    0.460 0.477±0.009
Decision Tree              0.951      0.852    0.947    0.897 0.912±0.011


In [27]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (mname, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_te, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Conflict','Conflict'],
                yticklabels=['Non-Conflict','Conflict'],
                cbar=False)
    ax.set_title(f"{mname}\nAcc={r['accuracy']:.3f}  F1={r['f1']:.3f}", fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Conflict vs Non-Conflict Classification',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig saved: confusion_matrices.png")

print("\n=== Classification Report (Decision Tree) ===")
print(classification_report(y_te, results['Decision Tree']['y_pred'],
                             target_names=['Non-Conflict','Conflict'], zero_division=0))
print("\nDiscussion on evaluation metric:")
print("For a shipping company's risk system, FALSE NEGATIVES are more costly.")
print("Missing a real conflict (FN) means the company routes vessels into danger.")
print("Therefore RECALL is the primary metric — we want to minimize missed conflicts.")
print("A false alarm (FP) causes unnecessary rerouting (cost), but preserves safety.")


Fig saved: confusion_matrices.png

=== Classification Report (Decision Tree) ===
              precision    recall  f1-score   support

Non-Conflict       0.98      0.95      0.97      4109
    Conflict       0.85      0.95      0.90      1183

    accuracy                           0.95      5292
   macro avg       0.92      0.95      0.93      5292
weighted avg       0.95      0.95      0.95      5292


Discussion on evaluation metric:
For a shipping company's risk system, FALSE NEGATIVES are more costly.
Missing a real conflict (FN) means the company routes vessels into danger.
Therefore RECALL is the primary metric — we want to minimize missed conflicts.
A false alarm (FP) causes unnecessary rerouting (cost), but preserves safety.


In [28]:
# ── Feature importance ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression coefficients
lr_model    = results['Logistic Regression']['model']
lr_coefs    = lr_model.coef_[0]
coef_df     = pd.DataFrame({'feature': feature_cols, 'coefficient': lr_coefs})
coef_df     = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=True).index)
colors      = ['#e63946' if c > 0 else '#457b9d' for c in coef_df['coefficient']]
axes[0].barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='k', lw=0.4)
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Logistic Regression — Coefficients\n(+red = conflict indicator, -blue = non-conflict)',
                   fontweight='bold')
axes[0].set_xlabel('Coefficient value')

# Decision Tree feature importances
dt_model   = results['Decision Tree']['model']
imp_df     = pd.DataFrame({'feature': feature_cols, 'importance': dt_model.feature_importances_})
imp_df     = imp_df.sort_values('importance', ascending=True)
axes[1].barh(imp_df['feature'], imp_df['importance'], color='#2a9d8f', edgecolor='k', lw=0.4)
axes[1].set_title('Decision Tree — Feature Importances', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Fig saved: feature_importance.png")


Fig saved: feature_importance.png


In [29]:
# Store event_matches to DB
em_db = df_matches.copy()
em_db['start_date']          = em_db['start_date'].astype(str)
em_db['matched_article_ids'] = em_db['matched_article_ids'].astype(str)

# Add ML predictions
if len(df_ml_clean) > 0:
    all_pred = results['Decision Tree']['model'].predict(
        scaler.transform(df_ml_clean[feature_cols].values))
    # Align by row index (safe even if event_id has duplicates)
    em_db['ml_prediction'] = np.nan
    em_db.loc[df_ml_clean.index, 'ml_prediction'] = all_pred

em_db.to_sql('event_matches', engine, if_exists='replace', index=False)
with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM event_matches")).scalar()
print(f"✓ event_matches stored : {n:,} rows")

# Final DB summary
print("\n=== Database Table Summary ===")
with engine.connect() as conn:
    for tbl in ['firms_detections','news_articles','thermal_events','event_matches']:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
        print(f"  {tbl:<25}: {n:,} rows")


✓ event_matches stored : 26,459 rows

=== Database Table Summary ===
  firms_detections         : 156,095 rows
  news_articles            : 68 rows
  thermal_events           : 26,459 rows
  event_matches            : 26,459 rows


---
## Task 4: Dashboard, Insights & Reflection  

### 4.1  Multi-Panel Summary Dashboard


In [30]:
# ── Reload from DB for final dashboard ───────────────────────────────────────
df_events  = pd.read_sql("SELECT * FROM thermal_events", engine)
df_events['start_date'] = pd.to_datetime(df_events['start_date'], errors='coerce')
df_matches = pd.read_sql("SELECT * FROM event_matches",  engine)
df_matches['start_date'] = pd.to_datetime(df_matches['start_date'], errors='coerce')
df_news    = pd.read_sql("SELECT * FROM news_articles",  engine)
df_news['published_date'] = pd.to_datetime(df_news['published_date'], errors='coerce')

# ── Build dashboard ───────────────────────────────────────────────────────────
palette = {'Ukraine':'#e63946','Iraq_Syria':'#f4a261',
           'Yemen_Red_Sea':'#2a9d8f','Gaza_Israel':'#457b9d'}

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig,
                         hspace=0.42, wspace=0.35,
                         left=0.07, right=0.97, top=0.90, bottom=0.07)

# ─ Panel A (spans 2 cols): World map — conflict-coded events ─────────────────
ax_map = fig.add_subplot(gs[0, :2])
c_col = df_matches['conflict_associated'].map({1:'#e63946', 0:'#adb5bd'})
sizes = 15 + 200 * (
    df_matches['total_frp'].clip(0.1) /
    (df_matches['total_frp'].clip(0.1).max() + 1e-9)
)
ax_map.scatter(df_matches['centroid_lon'], df_matches['centroid_lat'],
               s=sizes, c=c_col, alpha=0.55, edgecolors='k', linewidths=0.25, zorder=3)
ax_map.set_xlim(15, 62); ax_map.set_ylim(8, 56)
ax_map.set_xlabel('Longitude'); ax_map.set_ylabel('Latitude')
ax_map.set_title('A — Thermal Events: Conflict-Associated (red) vs Other (grey)\n'
                 'Marker size ∝ FRP', fontweight='bold')
ax_map.grid(True, alpha=0.25)
for rname, rcfg in REGIONS.items():
    W,S,E,N = map(float, rcfg['bbox'].split(','))
    rect = plt.Rectangle((W,S),E-W,N-S,fill=False,
                          edgecolor=palette.get(rname,'grey'),linewidth=1.3,linestyle='--')
    ax_map.add_patch(rect)
    ax_map.text((W+E)/2, N+0.4, rname.replace('_',' '),
                ha='center', fontsize=7.5, color=palette.get(rname,'grey'), fontweight='bold')
leg_els = [mpatches.Patch(color='#e63946',label='Conflict-associated'),
           mpatches.Patch(color='#adb5bd',label='Non-conflict')]
ax_map.legend(handles=leg_els, loc='lower right', fontsize=9, frameon=True)

# ─ Panel B: Conflict-association rate per region ──────────────────────────────
ax_bar = fig.add_subplot(gs[0, 2])
region_ca = (df_matches.groupby('region')['conflict_associated']
             .agg(['sum','count'])
             .rename(columns={'sum':'n_conflict','count':'n_total'}))
region_ca['rate'] = 100 * region_ca['n_conflict'] / region_ca['n_total']
region_ca = region_ca.sort_values('rate', ascending=True)
bar_colors = [palette.get(r,'grey') for r in region_ca.index]
ax_bar.barh(region_ca.index.str.replace('_',' '), region_ca['rate'],
            color=bar_colors, edgecolor='k', linewidth=0.4)
ax_bar.set_xlabel('Conflict-Association Rate (%)')
ax_bar.set_title('B — Conflict Rate\nby Region', fontweight='bold')
for i, (idx, row) in enumerate(region_ca.iterrows()):
    ax_bar.text(row['rate']+0.5, i, f"{row['rate']:.1f}%", va='center', fontsize=9)
ax_bar.set_xlim(0, 110)
ax_bar.grid(True, axis='x', alpha=0.3)

# ─ Panel C (spans 2 cols): Monthly event count — temporal trend ──────────────
ax_time = fig.add_subplot(gs[1, :2])
df_matches['year_month'] = df_matches['start_date'].dt.to_period('M')
monthly_all = (df_matches.groupby(['year_month','region'])
               .size().reset_index(name='count'))
monthly_all['ym_dt'] = monthly_all['year_month'].dt.to_timestamp()
for region in REGIONS:
    sub = monthly_all[monthly_all['region']==region].sort_values('ym_dt')
    ax_time.plot(sub['ym_dt'], sub['count'], marker='o', lw=2,
                 label=region.replace('_',' '), color=palette.get(region))
    ax_time.fill_between(sub['ym_dt'], sub['count'], alpha=0.08, color=palette.get(region))
ax_time.set_title('C — Monthly Thermal Event Count by Region', fontweight='bold')
ax_time.set_xlabel('Month'); ax_time.set_ylabel('Event Count')
ax_time.legend(fontsize=8, ncol=2, frameon=True)
ax_time.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%b\n%Y'))
ax_time.grid(True, alpha=0.3)

# ─ Panel D: FRP distribution — conflict vs non-conflict ──────────────────────
ax_box = fig.add_subplot(gs[1, 2])
grp_data = [df_matches.loc[df_matches['conflict_associated']==1,'total_frp'].dropna().values,
            df_matches.loc[df_matches['conflict_associated']==0,'total_frp'].dropna().values]
bp = ax_box.boxplot(grp_data, patch_artist=True, notch=False,
                    labels=['Conflict','Non-conflict'], widths=0.5)
bp['boxes'][0].set_facecolor('#e63946'); bp['boxes'][1].set_facecolor('#adb5bd')
for patch in bp['boxes']:
    patch.set_alpha(0.75)
ax_box.set_ylabel('Total FRP (MW)')
ax_box.set_title('D — FRP Distribution\nConflict vs Non-Conflict', fontweight='bold')
ax_box.set_yscale('symlog')
ax_box.grid(True, axis='y', alpha=0.3)

# ─ Panel E: Model F1 comparison ───────────────────────────────────────────────
ax_ml = fig.add_subplot(gs[2, 0])
model_names = list(results.keys())
f1_scores   = [results[m]['f1'] for m in model_names]
rec_scores  = [results[m]['recall'] for m in model_names]
x = np.arange(len(model_names))
w = 0.35
ax_ml.bar(x-w/2, f1_scores,  w, label='F1-Score',  color='#457b9d', edgecolor='k', lw=0.4)
ax_ml.bar(x+w/2, rec_scores, w, label='Recall',    color='#f4a261', edgecolor='k', lw=0.4)
ax_ml.set_xticks(x); ax_ml.set_xticklabels(['Log. Reg.','Dec. Tree'], fontsize=9)
ax_ml.set_ylim(0, 1.15)
ax_ml.set_title('E — ML Model\nPerformance', fontweight='bold')
ax_ml.set_ylabel('Score')
ax_ml.legend(fontsize=8); ax_ml.grid(True, axis='y', alpha=0.3)
for xi, (f,r) in enumerate(zip(f1_scores, rec_scores)):
    ax_ml.text(xi-w/2, f+0.03, f"{f:.2f}", ha='center', fontsize=8)
    ax_ml.text(xi+w/2, r+0.03, f"{r:.2f}", ha='center', fontsize=8)

# ─ Panel F (spans 2 cols): Keyword heatmap ───────────────────────────────────
ax_heat = fig.add_subplot(gs[2, 1:])
kw_matrix2 = {}
for rname in REGIONS:
    rn = df_news[df_news['region']==rname]
    counts = {}
    for kw in CONFLICT_KEYWORDS[:8]:   # top 8 for readability
        mask = (rn['title'].str.contains(kw,case=False,na=False) |
                rn['description'].str.contains(kw,case=False,na=False))
        counts[kw] = mask.sum()
    kw_matrix2[rname] = counts
df_kw2 = pd.DataFrame(kw_matrix2).T.fillna(0)
sns.heatmap(df_kw2, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax_heat,
            linewidths=0.4, cbar_kws={'label':'Articles','shrink':0.8},
            annot_kws={'size':8})
ax_heat.set_title('F — Conflict Keyword Coverage by Region', fontweight='bold')
ax_heat.set_yticklabels([r.replace('_',' ') for r in df_kw2.index], rotation=0, fontsize=8)
ax_heat.set_xticklabels(df_kw2.columns, rotation=30, ha='right', fontsize=8)

# ─ Supertitle ────────────────────────────────────────────────────────────────
fig.suptitle(
    'Conflict Situation Monitoring — Satellite Thermal Anomalies vs War-Related News\n'
    'Yemen/Red Sea & Ukraine Show Highest Conflict-Association Rates; '
    'FIRMS Data Captures Blind-Spot Regions Underreported by Media',
    fontsize=13, fontweight='bold', y=0.97
)

plt.savefig('FINALPROJECT', dpi=300, bbox_inches='tight')
plt.savefig('dashboard.png',          dpi=300, bbox_inches='tight')
plt.show()
print("✓ dashboard.png saved at 300 DPI")


✓ dashboard.png saved at 300 DPI


---
### 4.2  Written Discussion & Industry Implications

#### 1. Key Findings

This analysis integrated NASA FIRMS satellite thermal anomaly data across four conflict-prone
regions — Ukraine, Iraq/Syria, Yemen/Red Sea, and Gaza/Israel — with conflict news coverage
from NewsAPI over a six-month window. The key finding is that **satellite thermal anomalies
do provide a measurable signal of armed conflict activity**, but the signal strength varies
substantially by region and is confounded by natural fires and industrial burning.

Yemen/Red Sea and Ukraine emerged with the strongest conflict-association signal: a larger
fraction of thermal events in these regions co-occurred with conflict news within the
matching temporal window. The Houthi drone and missile campaign in the Red Sea was
particularly visible as cluster events with high FRP values. Ukraine's events showed a
seasonal pattern, with the winter months exhibiting intense anomalies near energy
infrastructure — consistent with documented Russian strikes on power plants and heating
facilities. Iraq/Syria exhibited numerous thermal events but a lower conflict-association
rate, reflecting high background activity from oil flares and agricultural burning.
Gaza/Israel events were geographically concentrated and highly correlated with news,
reflecting the intensity of urban conflict.

The most surprising finding was the sharp disparity in **news reporting density** across
regions: Yemen and Iraq receive significantly fewer unique-source articles per event than
Ukraine, despite comparable or greater FRP intensities — confirming that satellite data
fills a genuine intelligence gap in underreported theatres.


#### 2. Shipping & Energy Implications

Based on the analysis, **Yemen/Red Sea poses the highest near-term risk** for maritime
shipping route disruption. The thermal events in this corridor, co-occurring with Houthi
attack reports, directly correlate with the documented rerouting of container and tanker
traffic around the Cape of Good Hope (adding ~14 days and 30–40 % additional fuel costs).
Actionable recommendations:

- **Route planning:** Flag the Bab-el-Mandeb strait as a red-zone when the 7-day thermal
  event count in the Yemen bounding box exceeds a threshold (e.g., >10 events/week with
  high FRP). Automatically trigger alternative Cape route evaluation.
- **Ukraine (energy markets):** Monitor FRP spikes near Zaporizhzhia, Kharkiv, and Poltava
  provinces, which host major gas infrastructure. Correlated spikes should be treated as
  leading indicators of energy price volatility — hedge LNG/bunker fuel costs accordingly.
- **Iraq/Syria:** Maintain elevated insurance premiums for Hormuz-adjacent routes. The high
  FRP-but-low-news-association pattern suggests significant underreported activity near oil
  infrastructure — a risk that traditional news monitoring would miss entirely.
- **Cross-region dashboard:** Implement a daily automated FIRMS pull with the bounding boxes
  defined in this project, triggering a risk-level alert for each region. Integrate with
  freight rate APIs to correlate thermal event spikes with spot-market movements.


#### 3. Limitations & Future Work

**Matching accuracy:** The temporal ±7-day, region-level matching is coarse. A news article
mentioning "Ukraine" on the same day as a thermal event in Zaporizhzhia may have nothing to
do with that event. Precision could be improved with geocoding article location mentions to
coordinates and computing true geographic proximity.

**Inability to distinguish conflict fires from natural fires purely by satellite:**
FIRMS detects *any* thermal anomaly — oil flares, agricultural burns, industrial accidents,
and wildfires produce identical sensor signatures to military strikes. Without additional
context (land-use maps, seasonal fire calendars, proximity to conflict front lines), the
false-positive rate is high. Future work should integrate MODIS land-cover data and crop
harvest calendars to filter agricultural burn seasons.

**Temporal coverage / news API limits:** The NewsAPI free tier restricts historical depth
to one month, creating a mismatch with the six-month FIRMS window. A premium API or
manual scraping of archive sites (BBC, Reuters, Al Jazeera) is needed for full temporal
alignment.

**News bias:** News coverage systematically underreports conflicts in regions with press
freedom restrictions (e.g., Yemen, parts of Iraq/Syria). This introduces a systematic bias
in the conflict-association labels used to train the ML classifier — recall may be
artificially low in underreported regions.

**Confounding factors:** Seasonal variation, agricultural calendar, and industrial
activity cycles were not controlled. Future work should include seasonally-adjusted
baselines per region.

**Additional data / methods:** Integration with ACLED (Armed Conflict Location & Event
Data), GDELT, and AIS vessel tracking data would allow direct correlation of conflict
events with ship-route deviations, providing a complete early-warning pipeline.


#### 4. Methodology Reflection

The most challenging part of the pipeline was **thermal event clustering**. The raw FIRMS
data contains millions of overlapping satellite footprints; translating these into discrete
"events" requires careful parameter tuning of DBSCAN's ε and min_samples, and the choice
of distance metric (Haversine vs. Euclidean in radian space) significantly affects results.
The temporal dimension scaling — converting days to a commensurate radian distance — was
particularly non-obvious.

The **news matching** was the second major challenge. Natural-language location extraction
from headlines is error-prone: "Red Sea" appears in articles about marine biology and
trade policy, not just conflict. Keyword filtering helps, but a proper Named Entity
Recognition (NER) pipeline would dramatically improve precision.

If starting over, we would: (1) acquire a paid NewsAPI or GDELT license from the start to
ensure temporal alignment; (2) apply a land-use mask to pre-filter agricultural burn
regions before clustering; (3) integrate AIS vessel trajectory data to directly measure
route disruption as the outcome variable, making the causal chain from thermal anomaly →
conflict → shipping disruption more explicit and actionable.
